# TechOps Intelligence Platform
## Notebook 02 — Text Processing Pipeline

**Author:** Sudharshan
**Phase:** 2 — Document Processing  
**Goal:** Build production-grade text preprocessing pipeline
          that converts raw incident text into clean,
          structured, embedded chunks ready for ChromaDB

### What This Notebook Does
1. Load all text data sources
2. Clean and normalise text
3. Extract entities using spaCy NER
4. Chunk text intelligently
5. Generate embeddings (all-mpnet-base-v2, 768-dim)
6. Store in ChromaDB with metadata
7. Verify retrieval quality

### Data Sources Processed
- ServiceNow ITSM incidents (24,918)
- Real AI incidents (10,776)
- Cybersecurity incidents (10,000)
- Dan Luu postmortems (234)
- Generated postmortems (58)
- Incident response playbooks (174)

In [1]:
# SETUP + IMPORTS
# ─────────────────────────────────────────
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter

# NLP
import spacy
import re

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector DB
import chromadb
from chromadb.utils import embedding_functions

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain.text_splitter import RecursiveCharacterTextSplitter

# Progress bar
from tqdm import tqdm

# ─────────────────────────────────────────
# PROJECT PATHS
# ─────────────────────────────────────────
PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

RAW_TEXT    = PROJECT_ROOT / "data/raw/text"
PROCESSED   = PROJECT_ROOT / "data/processed"
EMBEDDINGS  = PROJECT_ROOT / "data/embeddings"
EMBEDDINGS.mkdir(parents=True, exist_ok=True)



## 1. Load spaCy NER Model
Used for extracting entities from incident text:
service names, error codes, IP addresses, thresholds

In [2]:
import spacy

# Load spaCy NER model
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded: en_core_web_sm")

# Test NER on sample incident
test_text = "PostgreSQL connection refused on port 5432 in us-east-1"
doc = nlp(test_text)

print(f"\nNER test on: '{test_text}'")
print("Entities found:")
for ent in doc.ents:
    print(f"  {ent.text:25} → {ent.label_}")

print(f"\nspaCy pipeline: {nlp.pipe_names}")


spaCy model loaded: en_core_web_sm

NER test on: 'PostgreSQL connection refused on port 5432 in us-east-1'
Entities found:
  5432                      → DATE

spaCy pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## 2. Text Cleaning Functions
Standardise all text before embedding
Handles: special chars, encoding issues,
         IT-specific patterns

In [3]:
# ─────────────────────────────────────────
# TEXT CLEANING PIPELINE
# ─────────────────────────────────────────

def clean_text(text: str) -> str:
    """
    Clean raw incident text for NLP processing.
    Preserves IT-specific patterns like error codes,
    IP addresses, and port numbers.
    """
    if not isinstance(text, str):
        return ""

    # Remove null bytes and control characters
    text = text.replace('\x00', '').replace('\r', ' ')

    # Normalise whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove URLs (keep domain for context)
    text = re.sub(
        r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+])+',
        '[URL]',
        text
    )

    # Preserve error codes (e.g. HTTP 500, ORA-12541)
    # Preserve IP addresses
    # Preserve port numbers (e.g. port 5432)
    # These are critical for BM25 keyword search

    # Remove excessive punctuation
    text = re.sub(r'[^\w\s\.\-\_\:\@\/\[\]]+', ' ', text)

    # Final whitespace cleanup
    text = text.strip()

    return text


def extract_entities(text: str, nlp_model) -> dict:
    """
    Extract IT-relevant entities from incident text.
    Returns structured dict of found entities.
    """
    if not text or len(text) < 10:
        return {}

    doc = nlp_model(text[:1000])  # cap for speed

    entities = {
        'services'   : [],
        'locations'  : [],
        'orgs'       : [],
        'error_codes': [],
        'ips'        : [],
        'ports'      : []
    }

    # spaCy entities
    for ent in doc.ents:
        if ent.label_ == 'ORG':
            entities['orgs'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'PRODUCT':
            entities['services'].append(ent.text)

    # Regex patterns for IT-specific entities
    # IP addresses
    ip_pattern = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'
    entities['ips'] = re.findall(ip_pattern, text)

    # Port numbers
    port_pattern = r'port\s+(\d+)'
    entities['ports'] = re.findall(port_pattern, text, re.IGNORECASE)

    # Error codes (e.g. HTTP 500, ORA-12541, ECONNREFUSED)
    error_pattern = r'\b(?:HTTP\s+\d{3}|[A-Z]{2,}-\d+|E[A-Z]{3,})\b'
    entities['error_codes'] = re.findall(error_pattern, text)

    # Remove duplicates
    for key in entities:
        entities[key] = list(set(entities[key]))

    return entities


def preprocess_incident(text: str, nlp_model) -> dict:
    """
    Full preprocessing pipeline for one incident.
    Returns cleaned text + extracted entities.
    """
    cleaned   = clean_text(text)
    entities  = extract_entities(cleaned, nlp_model)

    return {
        'original_text' : text,
        'cleaned_text'  : cleaned,
        'entities'      : entities,
        'char_count'    : len(cleaned),
        'word_count'    : len(cleaned.split())
    }


# ── Test The Pipeline ────────────────────
test_incidents = [
    "PostgreSQL connection refused on port 5432. Error: ECONNREFUSED",
    "HTTP 500 errors on payment-service. IP: 10.0.0.5 unreachable",
    "Kubernetes pod crashloopbackoff in namespace production"
]

print("=== Text Cleaning Test ===\n")
for text in test_incidents:
    result = preprocess_incident(text, nlp)
    print(f"Original : {result['original_text']}")
    print(f"Cleaned  : {result['cleaned_text']}")
    print(f"Entities : {result['entities']}")
    print(f"Words    : {result['word_count']}")
    print()

=== Text Cleaning Test ===

Original : PostgreSQL connection refused on port 5432. Error: ECONNREFUSED
Cleaned  : PostgreSQL connection refused on port 5432. Error: ECONNREFUSED
Entities : {'services': [], 'locations': [], 'orgs': [], 'error_codes': ['ECONNREFUSED'], 'ips': [], 'ports': ['5432']}
Words    : 8

Original : HTTP 500 errors on payment-service. IP: 10.0.0.5 unreachable
Cleaned  : HTTP 500 errors on payment-service. IP: 10.0.0.5 unreachable
Entities : {'services': [], 'locations': [], 'orgs': ['IP'], 'error_codes': ['HTTP 500'], 'ips': ['10.0.0.5'], 'ports': []}
Words    : 8

Original : Kubernetes pod crashloopbackoff in namespace production
Cleaned  : Kubernetes pod crashloopbackoff in namespace production
Entities : {'services': [], 'locations': [], 'orgs': [], 'error_codes': [], 'ips': [], 'ports': []}
Words    : 6



## 3. Load Embedding Model
all-mpnet-base-v2 — 768 dimensions
Runs on CPU to preserve GPU VRAM for Qwen2.5

In [4]:
# ─────────────────────────────────────────
# LOAD EMBEDDING MODEL
# all-mpnet-base-v2 — 768 dimensions
# Better semantic understanding than MiniLM
# ─────────────────────────────────────────
print("Loading embedding model...")
print("(First run downloads ~420MB — subsequent runs are instant)")

embedding_model = SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2',
    device='cpu'   # keep GPU free for Qwen2.5
)

print(f"✓ Model loaded")
print(f"  Embedding dimensions : 768")
print(f"  Device               : CPU")

# Test embedding
test_sentences = [
    "Database connection refused on port 5432",
    "PostgreSQL is down and not accepting connections",
    "The weather is nice today"
]

embeddings = embedding_model.encode(test_sentences)
print(f"\nEmbedding test:")
print(f"  Input sentences : {len(test_sentences)}")
print(f"  Output shape    : {embeddings.shape}")

# Verify semantic similarity
from sklearn.metrics.pairwise import cosine_similarity
sim_12 = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
sim_13 = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]

print(f"\nSimilarity test:")
print(f"  DB refused vs DB down   : {sim_12:.3f}")
print(f"  DB refused vs weather   : {sim_13:.3f}")
print(f"\nSemantic similarity working correctly" if sim_12 > sim_13
      else "\n  ⚠️  Check embedding model")

Loading embedding model...
(First run downloads ~420MB — subsequent runs are instant)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5202.88it/s]


✓ Model loaded
  Embedding dimensions : 768
  Device               : CPU

Embedding test:
  Input sentences : 3
  Output shape    : (3, 768)

Similarity test:
  DB refused vs DB down   : 0.547
  DB refused vs weather   : 0.034

Semantic similarity working correctly


## 4. Setup ChromaDB
Unified vector store for all text sources
Each source gets its own collection
with document type metadata for filtered retrieval

In [5]:
# ─────────────────────────────────────────
# CHROMADB SETUP
# Persistent storage — survives restarts
# ─────────────────────────────────────────
chroma_path = str(EMBEDDINGS / "chroma_db")

client = chromadb.PersistentClient(path=chroma_path)

print(f"✓ ChromaDB initialised")
print(f"  Storage path: {chroma_path}")

# Create collections — one per document type
# Using cosine similarity (better for text)
collections = {}
collection_names = [
    "incidents",      # all incident records
    "postmortems",    # postmortem docs
    "playbooks",      # response playbooks
    "logs",           # system log patterns
]

for name in collection_names:
    collections[name] = client.get_or_create_collection(
        name     = name,
        metadata = {"hnsw:space": "cosine"}
    )
    print(f"  ✓ Collection '{name}' ready")

print(f"\nExisting collections: {[c.name for c in client.list_collections()]}")

✓ ChromaDB initialised
  Storage path: C:\Users\sudha\techops-intelligence\data\embeddings\chroma_db
  ✓ Collection 'incidents' ready
  ✓ Collection 'postmortems' ready
  ✓ Collection 'playbooks' ready
  ✓ Collection 'logs' ready

Existing collections: ['logs', 'playbooks', 'incidents', 'postmortems']


## 5. Process + Embed Incidents
Process all incident sources and store in ChromaDB
Sources: ServiceNow + Real AI incidents + Cybersecurity

In [6]:
# ─────────────────────────────────────────
# PROCESS + EMBED ALL INCIDENTS
# ─────────────────────────────────────────
BATCH_SIZE = 64   # optimal for your 16GB RAM

def embed_and_store(
    texts     : list,
    metadatas : list,
    ids       : list,
    collection,
    model,
    batch_size: int = 64
):
    """
    Embed texts in batches and store in ChromaDB.
    Handles large datasets without memory issues.
    """
    total   = len(texts)
    stored  = 0
    errors  = 0

    for i in tqdm(
        range(0, total, batch_size),
        desc=f"Embedding → {collection.name}"
    ):
        batch_texts  = texts[i:i+batch_size]
        batch_meta   = metadatas[i:i+batch_size]
        batch_ids    = ids[i:i+batch_size]

        # Skip empty texts
        valid = [
            (t, m, id_)
            for t, m, id_ in zip(batch_texts, batch_meta, batch_ids)
            if t and len(t.strip()) > 10
        ]

        if not valid:
            continue

        v_texts, v_meta, v_ids = zip(*valid)

        try:
            embeddings = model.encode(
                list(v_texts),
                show_progress_bar=False,
                batch_size=batch_size
            )

            collection.add(
                documents  = list(v_texts),
                embeddings = embeddings.tolist(),
                metadatas  = list(v_meta),
                ids        = list(v_ids)
            )
            stored += len(v_texts)

        except Exception as e:
            errors += 1
            print(f"  Batch error at {i}: {e}")

    return stored, errors


# ── Load Processed Incidents ──────────────
print("Loading processed incident datasets...")

# ServiceNow ITSM
df_itsm = pd.read_csv(PROCESSED / "incident_event_log_clean.csv")
print(f"ServiceNow ITSM    : {len(df_itsm):,} rows")

# Real AI Incidents
df_real = pd.read_csv(PROCESSED / "real_incidents_clean.csv")
print(f"Real AI Incidents  : {len(df_real):,} rows")

# Cybersecurity
df_cyber = pd.read_csv(PROCESSED / "cybersecurity_clean.csv")
print(f"Cybersecurity      : {len(df_cyber):,} rows")


Loading processed incident datasets...
ServiceNow ITSM    : 24,918 rows
Real AI Incidents  : 10,776 rows
Cybersecurity      : 10,000 rows


In [7]:
# ─────────────────────────────────────────
# EMBED SERVICENOW ITSM INCIDENTS (upsert into same collection)
# ─────────────────────────────────────────
print("Processing ServiceNow ITSM incidents...")

# Build text from available columns
def build_itsm_text(row):
    parts = []
    if pd.notna(row.get('category')):
        parts.append(f"Category: {row['category']}")
    if pd.notna(row.get('subcategory')):
        parts.append(f"Subcategory: {row['subcategory']}")
    if pd.notna(row.get('symptom')):
        parts.append(f"Symptom: {row['symptom']}")
    if pd.notna(row.get('priority_clean')):
        parts.append(f"Priority: {row['priority_clean']}")
    if pd.notna(row.get('impact_clean')):
        parts.append(f"Impact: {row['impact_clean']}")
    return ". ".join(parts) if parts else "IT incident"

df_itsm['embed_text'] = df_itsm.apply(build_itsm_text, axis=1)

texts     = df_itsm['embed_text'].tolist()
ids       = [f"itsm_{i}" for i in range(len(df_itsm))]
metadatas = [
    {
        "source"  : "servicenow_itsm",
        "priority": str(row.get('priority_clean', 'P3')),
        "category": str(row.get('category', '')),
        "doc_type": "incident"
    }
    for _, row in df_itsm.iterrows()
]

# ── Upsert logic: only embed new IDs ───────────
existing = collections['incidents'].get(ids=ids)
existing_ids = set(existing["ids"]) if existing and "ids" in existing else set()

new_texts, new_ids, new_metadatas = [], [], []
for text, id_, meta in zip(texts, ids, metadatas):
    if id_ not in existing_ids:
        new_texts.append(text)
        new_ids.append(id_)
        new_metadatas.append(meta)

if new_ids:
    stored, errors = embed_and_store(
        new_texts, new_metadatas, new_ids,
        collections['incidents'],
        embedding_model,
        BATCH_SIZE
    )
    print(f"\n✓ Added {stored:,} new ITSM incidents")
    print(f"  Errors : {errors}")
else:
    print("\nNo new ITSM incidents to embed")

print(f"  Collection count: {collections['incidents'].count()}")


Processing ServiceNow ITSM incidents...


Embedding → incidents: 100%|██████████| 390/390 [41:12<00:00,  6.34s/it]


✓ Added 24,918 new ITSM incidents
  Errors : 0
  Collection count: 24918


In [8]:
# ─────────────────────────────────────────
# EMBED REAL AI INCIDENTS (upsert into same collection)
# ─────────────────────────────────────────
print("Processing Real AI incidents...")

texts     = df_real['incident_text'].fillna('').tolist()
ids       = [f"real_{i}" for i in range(len(df_real))]
metadatas = [
    {
        "source"    : "real_ai_incidents",
        "priority"  : str(row.get('priority_clean', 'P3')),
        "provider"  : str(row.get('provider_name', '')),
        "category"  : str(row.get('category_clean', '')),
        "duration"  : str(row.get('duration_minutes', 0)),
        "doc_type"  : "incident"
    }
    for _, row in df_real.iterrows()
]

# ── Upsert logic: only embed new IDs ───────────
existing = collections['incidents'].get(ids=ids)
existing_ids = set(existing["ids"]) if existing and "ids" in existing else set()

new_texts, new_ids, new_metadatas = [], [], []
for text, id_, meta in zip(texts, ids, metadatas):
    if id_ not in existing_ids:
        new_texts.append(text)
        new_ids.append(id_)
        new_metadatas.append(meta)

if new_ids:
    stored, errors = embed_and_store(
        new_texts, new_metadatas, new_ids,
        collections['incidents'],
        embedding_model,
        BATCH_SIZE
    )
    print(f"\nAdded {stored:,} new Real AI incidents")
    print(f"  Errors : {errors}")
else:
    print("\nNo new Real AI incidents to embed")

print(f"  Collection count: {collections['incidents'].count()}")


Processing Real AI incidents...


Embedding → incidents: 100%|██████████| 169/169 [23:07<00:00,  8.21s/it]


Added 10,776 new Real AI incidents
  Errors : 0
  Collection count: 35694


In [9]:
# ─────────────────────────────────────────
# EMBED CYBERSECURITY INCIDENTS (upsert into same collection)
# ─────────────────────────────────────────
print("Processing cybersecurity incidents...")

texts     = df_cyber['incident_text'].fillna('').tolist()
ids       = [f"cyber_{i}" for i in range(len(df_cyber))]
metadatas = [
    {
        "source"     : "cybersecurity",
        "priority"   : str(row.get('priority_clean', 'P3')),
        "category"   : "security",
        "attack_type": str(row.get('attack_type', '')),
        "doc_type"   : "incident"
    }
    for _, row in df_cyber.iterrows()
]

# ── Upsert logic: only embed new IDs ───────────
existing = collections['incidents'].get(ids=ids)
existing_ids = set(existing["ids"]) if existing and "ids" in existing else set()

new_texts, new_ids, new_metadatas = [], [], []
for text, id_, meta in zip(texts, ids, metadatas):
    if id_ not in existing_ids:
        new_texts.append(text)
        new_ids.append(id_)
        new_metadatas.append(meta)

if new_ids:
    stored, errors = embed_and_store(
        new_texts, new_metadatas, new_ids,
        collections['incidents'],
        embedding_model,
        BATCH_SIZE
    )
    print(f"\nAdded {stored:,} new cybersecurity incidents")
    print(f"  Errors : {errors}")
else:
    print("\nNo new cybersecurity incidents to embed")

print(f"\nTotal incidents in ChromaDB: {collections['incidents'].count():,}")


Processing cybersecurity incidents...


Embedding → incidents: 100%|██████████| 157/157 [13:50<00:00,  5.29s/it]


Added 10,000 new cybersecurity incidents
  Errors : 0

Total incidents in ChromaDB: 45,694


## 6. Process + Embed Postmortems
Both generated postmortems and Dan Luu real postmortems
These are the PRIMARY RAG knowledge source for diagnosis

In [10]:
# ─────────────────────────────────────────
# EMBED POSTMORTEMS — BOTH SOURCES (upsert)
# ─────────────────────────────────────────

# ── Source 1: Generated Postmortems ──────
print("Processing generated postmortems...")

with open(RAW_TEXT / "postmortems/postmortems.json", 'r') as f:
    generated_pms = json.load(f)

gen_texts, gen_ids, gen_meta = [], [], []

for i, pm in enumerate(generated_pms):
    text_parts = []
    if pm.get('title'):
        text_parts.append(f"Incident: {pm['title']}")
    if pm.get('summary'):
        text_parts.append(f"Summary: {pm['summary']}")
    if pm.get('root_cause'):
        text_parts.append(f"Root cause: {pm['root_cause']}")
    if pm.get('resolution_steps'):
        steps = ". ".join(pm['resolution_steps'])
        text_parts.append(f"Resolution: {steps}")
    if pm.get('lessons_learned'):
        lessons = ". ".join(pm['lessons_learned'])
        text_parts.append(f"Lessons: {lessons}")

    full_text = " | ".join(text_parts)
    clean     = clean_text(full_text)

    if len(clean) > 50:
        gen_texts.append(clean)
        gen_ids.append(f"gen_pm_{i}")
        gen_meta.append({
            "source"  : "generated_postmortem",
            "category": pm.get('category', ''),
            "severity": pm.get('severity', ''),
            "doc_type": "postmortem"
        })

# Upsert logic for generated postmortems
existing = collections['postmortems'].get(ids=gen_ids)
existing_ids = set(existing["ids"]) if existing and "ids" in existing else set()

new_texts, new_ids, new_meta = [], [], []
for text, id_, meta in zip(gen_texts, gen_ids, gen_meta):
    if id_ not in existing_ids:
        new_texts.append(text)
        new_ids.append(id_)
        new_meta.append(meta)

if new_ids:
    stored_gen, _ = embed_and_store(
        new_texts, new_meta, new_ids,
        collections['postmortems'],
        embedding_model
    )
    print(f"Added {stored_gen} new generated postmortems")
else:
    print("No new generated postmortems to embed")

# ── Source 2: Dan Luu Postmortems ────────
print("\nProcessing Dan Luu postmortems...")

with open(PROCESSED / "danluu_postmortems.json", 'r') as f:
    danluu_pms = json.load(f)

dl_texts, dl_ids, dl_meta = [], [], []

for i, pm in enumerate(danluu_pms):
    clean = clean_text(pm.get('full_text', ''))
    if len(clean) > 50:
        dl_texts.append(clean)
        dl_ids.append(f"danluu_{i}")
        dl_meta.append({
            "source"  : "danluu_postmortem",
            "company" : pm.get('source', ''),
            "url"     : pm.get('url', ''),
            "doc_type": "postmortem"
        })

# Upsert logic for Dan Luu postmortems
existing = collections['postmortems'].get(ids=dl_ids)
existing_ids = set(existing["ids"]) if existing and "ids" in existing else set()

new_texts, new_ids, new_meta = [], [], []
for text, id_, meta in zip(dl_texts, dl_ids, dl_meta):
    if id_ not in existing_ids:
        new_texts.append(text)
        new_ids.append(id_)
        new_meta.append(meta)

if new_ids:
    stored_dl, _ = embed_and_store(
        new_texts, new_meta, new_ids,
        collections['postmortems'],
        embedding_model
    )
    print(f"Added {stored_dl} new Dan Luu postmortems")
else:
    print("No new Dan Luu postmortems to embed")

print(f"\nTotal postmortems in ChromaDB: {collections['postmortems'].count()}")


Processing generated postmortems...


Embedding → postmortems:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding → postmortems: 100%|██████████| 1/1 [00:31<00:00, 31.09s/it]


Added 58 new generated postmortems

Processing Dan Luu postmortems...


Embedding → postmortems: 100%|██████████| 4/4 [04:44<00:00, 71.05s/it]

Added 234 new Dan Luu postmortems

Total postmortems in ChromaDB: 292


In [11]:
# ─────────────────────────────────────────
# EMBED INCIDENT RESPONSE PLAYBOOKS (upsert into same collection)
# ─────────────────────────────────────────
print("Processing incident response playbooks...")

playbook_path = RAW_TEXT / "playbooks/incident_response_playbook_dataset.jsonl"

pb_texts, pb_ids, pb_meta = [], [], []

with open(playbook_path, 'r') as f:
    for i, line in enumerate(f):
        pb = json.loads(line)

        # Build rich text from playbook
        text_parts = []

        if pb.get('incident_type'):
            text_parts.append(f"Incident type: {pb['incident_type']}")
        if pb.get('target_asset'):
            text_parts.append(f"Target: {pb['target_asset']}")
        if pb.get('initial_vector'):
            text_parts.append(f"Vector: {pb['initial_vector']}")

        # Extract playbook steps
        if pb.get('playbook_steps'):
            steps = pb['playbook_steps']
            if isinstance(steps, list):
                step_texts = []
                for step in steps:
                    if isinstance(step, dict):
                        phase  = step.get('phase', '')
                        action = step.get('action', '')
                        if phase and action:
                            step_texts.append(f"{phase}: {action}")
                if step_texts:
                    text_parts.append("Steps: " + ". ".join(step_texts[:5]))

        if pb.get('tags'):
            tags = pb['tags'] if isinstance(pb['tags'], list) else []
            text_parts.append(f"Tags: {', '.join(tags)}")

        full_text = " | ".join(text_parts)
        clean     = clean_text(full_text)

        if len(clean) > 50:
            pb_texts.append(clean)
            pb_ids.append(f"playbook_{i}")
            pb_meta.append({
                "source"        : "incident_playbook",
                "incident_type" : pb.get('incident_type', ''),
                "severity"      : pb.get('severity', ''),
                "doc_type"      : "playbook"
            })

# ── Upsert logic: only embed new IDs ───────────
existing = collections['playbooks'].get(ids=pb_ids)
existing_ids = set(existing["ids"]) if existing and "ids" in existing else set()

new_texts, new_ids, new_meta = [], [], []
for text, id_, meta in zip(pb_texts, pb_ids, pb_meta):
    if id_ not in existing_ids:
        new_texts.append(text)
        new_ids.append(id_)
        new_meta.append(meta)

if new_ids:
    stored_pb, _ = embed_and_store(
        new_texts, new_meta, new_ids,
        collections['playbooks'],
        embedding_model
    )
    print(f"Added {stored_pb} new playbooks")
else:
    print("No new playbooks to embed")

print(f"Total playbooks in ChromaDB: {collections['playbooks'].count()}")


Processing incident response playbooks...


Embedding → playbooks:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding → playbooks: 100%|██████████| 3/3 [00:43<00:00, 14.50s/it]

Added 174 new playbooks
Total playbooks in ChromaDB: 174


## 7. Retrieval Quality Test
Verify embeddings produce meaningful search results
Test queries that mimic real agent requests

In [12]:
# ─────────────────────────────────────────
# RETRIEVAL QUALITY TEST
# Simulate what agents will actually query
# ─────────────────────────────────────────

def semantic_search(query, collection, model, n=3):
    """Simple semantic search on ChromaDB collection"""
    query_embedding = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = n,
        include          = ['documents', 'metadatas', 'distances']
    )
    return results

test_queries = [
    {
        "query"      : "database connection refused PostgreSQL",
        "collection" : "incidents",
        "expected"   : "database related incidents"
    },
    {
        "query"      : "kubernetes pod crashloopbackoff OOMKilled",
        "collection" : "incidents",
        "expected"   : "kubernetes incidents"
    },
    {
        "query"      : "what is the root cause of memory exhaustion",
        "collection" : "postmortems",
        "expected"   : "memory postmortem"
    },
    {
        "query"      : "how to respond to ransomware attack",
        "collection" : "playbooks",
        "expected"   : "security playbook"
    },
    {
        "query"      : "DNS resolution failure service unreachable",
        "collection" : "incidents",
        "expected"   : "network incidents"
    }
]

print("=== Retrieval Quality Test ===\n")
for test in test_queries:
    results = semantic_search(
        test['query'],
        collections[test['collection']],
        embedding_model,
        n=2
    )

    print(f"Query      : {test['query']}")
    print(f"Expected   : {test['expected']}")
    print(f"Collection : {test['collection']}")
    print(f"Results    :")

    docs      = results['documents'][0]
    distances = results['distances'][0]
    metas     = results['metadatas'][0]

    for doc, dist, meta in zip(docs, distances, metas):
        score = 1 - dist  # cosine distance → similarity
        print(f"  Score: {score:.3f} | Source: {meta.get('source','')}")
        print(f"  Text : {doc[:120]}...")
    print()

=== Retrieval Quality Test ===

Query      : database connection refused PostgreSQL
Expected   : database related incidents
Collection : incidents
Results    :
  Score: 0.249 | Source: cybersecurity
  Text : Security incident: Brute Force targeting Database in Finance industry. Outcome: Failure. Mitigated by: Block IP....
  Score: 0.249 | Source: cybersecurity
  Text : Security incident: Brute Force targeting Database in Finance industry. Outcome: Failure. Mitigated by: Block IP....

Query      : kubernetes pod crashloopbackoff OOMKilled
Expected   : kubernetes incidents
Collection : incidents
Results    :
  Score: 0.557 | Source: real_ai_incidents
  Text : Managed Kubernetes Service. Provider: DigitalOcean. Category: infrastructure. Severity: minor. Duration: 2880 minutes....
  Score: 0.540 | Source: real_ai_incidents
  Text : Kubernetes Clusters - Authentication Errors. Provider: DigitalOcean. Category: infrastructure. Severity: major. Duration...

Query      : what is the root cause

In [13]:
# ─────────────────────────────────────────
# NOTEBOOK 02 — FINAL SUMMARY
# ─────────────────────────────────────────
print("   NOTEBOOK 02 — TEXT PIPELINE COMPLETE")
print("=" * 55)

total_docs = sum(
    collections[name].count()
    for name in collection_names
)

for name in collection_names:
    count = collections[name].count()
    print(f"  {name:20} : {count:,} documents")

print(f"  {'─'*35}")
print(f"  {'Total':20} : {total_docs:,} documents")
print()
print(f"  Embedding model  : all-mpnet-base-v2")
print(f"  Dimensions       : 768")
print(f"  Storage          : {chroma_path}")

   NOTEBOOK 02 — TEXT PIPELINE COMPLETE
  incidents            : 45,694 documents
  postmortems          : 292 documents
  playbooks            : 174 documents
  logs                 : 0 documents
  ───────────────────────────────────
  Total                : 46,160 documents

  Embedding model  : all-mpnet-base-v2
  Dimensions       : 768
  Storage          : C:\Users\sudha\techops-intelligence\data\embeddings\chroma_db


In [14]:
# ─────────────────────────────────────────
# FIX — DELETE ITSM FROM INCIDENTS COLLECTION
# ITSM has no real text — hurts retrieval quality
# Keep only real_ai_incidents + cybersecurity
# ─────────────────────────────────────────

# Get all ITSM IDs
print("Removing ITSM documents from incidents collection...")

# Get count before
before_count = collections['incidents'].count()
print(f"Before: {before_count:,} documents")

# Delete ITSM documents by ID prefix
# ChromaDB doesn't support prefix delete
# so we query and delete in batches
itsm_ids = [f"itsm_{i}" for i in range(len(df_itsm))]

# Delete in batches of 1000
batch_size = 1000
deleted    = 0
for i in range(0, len(itsm_ids), batch_size):
    batch = itsm_ids[i:i+batch_size]
    try:
        collections['incidents'].delete(ids=batch)
        deleted += len(batch)
    except Exception as e:
        pass

after_count = collections['incidents'].count()
print(f"After  : {after_count:,} documents")
print(f"Removed: {before_count - after_count:,} ITSM documents")
print(f"\n✓ Incidents collection now contains only real text")

Removing ITSM documents from incidents collection...
Before: 45,694 documents
After  : 20,776 documents
Removed: 24,918 ITSM documents

✓ Incidents collection now contains only real text


In [15]:
# Re-run Query 1 after removing ITSM
results = semantic_search(
    "database connection refused PostgreSQL",
    collections['incidents'],
    embedding_model,
    n=3
)

print("Query: database connection refused PostgreSQL")
print("Results after fix:")
for doc, dist, meta in zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
):
    score = 1 - dist
    print(f"  Score: {score:.3f} | Source: {meta.get('source','')}")
    print(f"  Text : {doc[:120]}...")

Query: database connection refused PostgreSQL
Results after fix:
  Score: 0.249 | Source: cybersecurity
  Text : Security incident: Brute Force targeting Database in Finance industry. Outcome: Failure. Mitigated by: Block IP....
  Score: 0.249 | Source: cybersecurity
  Text : Security incident: Brute Force targeting Database in Finance industry. Outcome: Failure. Mitigated by: Block IP....
  Score: 0.247 | Source: cybersecurity
  Text : Security incident: Brute Force targeting Database in Manufacturing industry. Outcome: Failure. Mitigated by: Block IP....


In [16]:
# ─────────────────────────────────────────
# TEST QUERY AGAINST POSTMORTEMS
# Database connection incidents live here
# not in the incidents collection
# ─────────────────────────────────────────
queries = [
    "database connection refused PostgreSQL port 5432",
    "memory exhaustion OutOfMemoryError",
    "kubernetes pod crashloop OOMKilled",
    "network DNS resolution failure",
    "disk full ENOSPC storage",
    "how to resolve API timeout cascade"
]

print("=== Postmortem Collection Retrieval Test ===\n")
for query in queries:
    results = semantic_search(
        query,
        collections['postmortems'],
        embedding_model,
        n=2
    )
    print(f"Query: {query}")
    for doc, dist, meta in zip(
        results['documents'][0],
        results['distances'][0],
        results['metadatas'][0]
    ):
        score = 1 - dist
        print(f"  {score:.3f} | {meta.get('source','')} | {doc[:100]}...")
    print()

=== Postmortem Collection Retrieval Test ===

Query: database connection refused PostgreSQL port 5432
  0.756 | generated_postmortem | Incident: PostgreSQL Connection Refusal Incident   Summary: A critical connection refusal incident o...
  0.480 | generated_postmortem | Incident: PostgreSQL Max Connections Reached   Summary: The PostgreSQL database reached its max_conn...

Query: memory exhaustion OutOfMemoryError
  0.479 | generated_postmortem | Incident: Memory Swap Exhaustion Incident   Summary: Swap usage reached 98  due to excessive memory ...
  0.469 | generated_postmortem | Incident: Memory Exhaustion Incident on Application Server   Summary: The Application Server experie...

Query: kubernetes pod crashloop OOMKilled
  0.808 | generated_postmortem | Incident: Kubernetes Pod OOMKilled Incident   Summary: A Kubernetes Pod experienced an OOMKilled eve...
  0.653 | generated_postmortem | Incident: CPU and Memory Quota Exceeded in Kubernetes Cluster   Summary: The Kubernetes schedu

In [18]:
# ─────────────────────────────────────────
# IMPROVE CYBERSECURITY INCIDENT TEXT
# Make it more specific and searchable
# ─────────────────────────────────────────
df_cyber = pd.read_csv(PROCESSED / "cybersecurity_clean.csv")

# Better text format
df_cyber['incident_text'] = (
    df_cyber['attack_type'] + " attack on " +
    df_cyber['target_system'] + ". " +
    "Industry: " + df_cyber['industry'] + ". " +
    "Outcome: " + df_cyber['outcome'] + ". " +
    "Mitigation: " + df_cyber['mitigation_method'] + ". " +
    "Response time: " + df_cyber['response_time_min'].astype(str) + " minutes."
)

print("Sample improved cybersecurity text:")
for text in df_cyber['incident_text'].head(3):
    print(f"  → {text}")

# Save improved version
df_cyber.to_csv(PROCESSED / "cybersecurity_clean.csv", index=False)
print(f"\nImproved cybersecurity text saved")

# Re-embed cybersecurity with better text
print("\nRe-embedding cybersecurity incidents...")

# Delete old cybersecurity from ChromaDB
cyber_ids = [f"cyber_{i}" for i in range(len(df_cyber))]
try:
    collections['incidents'].delete(ids=cyber_ids)
except:
    pass

# Re-add with better text
texts     = df_cyber['incident_text'].fillna('').tolist()
ids       = cyber_ids
metadatas = [
    {
        "source"     : "cybersecurity",
        "priority"   : str(row.get('priority_clean', 'P3')),
        "category"   : "security",
        "attack_type": str(row.get('attack_type', '')),
        "doc_type"   : "incident"
    }
    for _, row in df_cyber.iterrows()
]

stored, errors = embed_and_store(
    texts, metadatas, ids,
    collections['incidents'],
    embedding_model,
    BATCH_SIZE
)
print(f"Cybersecurity re-embedded: {stored:,} documents")

Sample improved cybersecurity text:
  → Phishing attack on User Account. Industry: Education. Outcome: Failure. Mitigation: Patch. Response time: 138 minutes.
  → SQL Injection attack on Web Server. Industry: Finance. Outcome: Success. Mitigation: Block IP. Response time: 42 minutes.
  → Cross-Site Scripting attack on Email Server. Industry: Retail. Outcome: Success. Mitigation: Patch. Response time: 92 minutes.

Improved cybersecurity text saved

Re-embedding cybersecurity incidents...


Embedding → incidents: 100%|██████████| 157/157 [05:15<00:00,  2.01s/it]

Cybersecurity re-embedded: 10,000 documents


In [19]:
# ─────────────────────────────────────────
# NOTEBOOK 02 — FINAL COLLECTION SUMMARY
# ─────────────────────────────────────────
print("=" * 55)
print("   NOTEBOOK 02 — TEXT PIPELINE COMPLETE")
print("=" * 55)

collection_summary = {}
for name in collection_names:
    count = collections[name].count()
    collection_summary[name] = count
    print(f"  {name:20} : {count:,} documents")

total = sum(collection_summary.values())
print(f"  {'─'*35}")
print(f"  {'Total':20} : {total:,} documents")
print()
print("  Retrieval Quality (postmortems):")
print("  Min score : 0.415")
print("  Max score : 0.808")
print("  Avg score : ~0.57")
print()
print("  Embedding model : all-mpnet-base-v2 (768-dim)")
print(f"  Storage         : {chroma_path}")

   NOTEBOOK 02 — TEXT PIPELINE COMPLETE
  incidents            : 20,776 documents
  postmortems          : 292 documents
  playbooks            : 174 documents
  logs                 : 0 documents
  ───────────────────────────────────
  Total                : 21,242 documents

  Retrieval Quality (postmortems):
  Min score : 0.415
  Max score : 0.808
  Avg score : ~0.57

  Embedding model : all-mpnet-base-v2 (768-dim)
  Storage         : C:\Users\sudha\techops-intelligence\data\embeddings\chroma_db
